In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("RetailLakehouse_SilverToGold") \
    .config("spark.jars.packages",
            "io.delta:delta-spark_2.12:3.1.0,org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint",          "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key",        "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key",        "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"Session active: {spark.sparkContext.appName}")

Spark version : 3.5.0
Session active: RetailLakehouse_SilverToGold


In [ ]:
# ─────────────────────────────────────────
# READ SILVER DELTA TABLES
# ─────────────────────────────────────────
spark.read.format("delta").load("s3a://silver/delta/transactions") \
    .createOrReplaceTempView("silver_transactions")

spark.read.format("delta").load("s3a://silver/delta/customers") \
    .createOrReplaceTempView("silver_customers")

spark.read.format("delta").load("s3a://silver/delta/products") \
    .createOrReplaceTempView("silver_products")

spark.read.format("delta").load("s3a://silver/delta/stores") \
    .createOrReplaceTempView("silver_stores")

# Verify all views registered
for view in ["silver_transactions","silver_customers",
             "silver_products","silver_stores"]:
    count = spark.sql(f"SELECT COUNT(*) FROM {view}").collect()[0][0]
    print(f"{view}: {count:,} rows")

✔ silver_transactions: 500,000 rows
✔ silver_customers: 10,000 rows
✔ silver_products: 1,000 rows
✔ silver_stores: 200 rows


In [3]:
# ─────────────────────────────────────────
# DATA QUALITY CHECKS BEFORE GOLD WRITE
# ─────────────────────────────────────────
spark.sql("""
    SELECT
        COUNT(*)                                    AS total_rows,
        COUNT(DISTINCT customer_id)                 AS unique_customers,
        COUNT(DISTINCT product_id)                  AS unique_products,
        COUNT(DISTINCT store_id)                    AS unique_stores,
        MIN(transaction_date)                       AS earliest_date,
        MAX(transaction_date)                       AS latest_date,
        SUM(CASE WHEN net_revenue < 0 THEN 1 END)   AS negative_revenue_count,
        SUM(CASE WHEN quantity <= 0 THEN 1 END)     AS invalid_quantity_count
    FROM silver_transactions
""").show()

+----------+----------------+---------------+-------------+-------------+-----------+----------------------+----------------------+
|total_rows|unique_customers|unique_products|unique_stores|earliest_date|latest_date|negative_revenue_count|invalid_quantity_count|
+----------+----------------+---------------+-------------+-------------+-----------+----------------------+----------------------+
|    500000|           10000|           1000|          200|   2024-01-01| 2024-12-31|                   751|                  NULL|
+----------+----------------+---------------+-------------+-------------+-----------+----------------------+----------------------+



In [ ]:
# ─────────────────────────────────────────
# GOLD: FACT SALES AGGREGATED
# ─────────────────────────────────────────
df_gold_fact = spark.sql("""
    SELECT
        t.transaction_id                                AS sales_key,
        CAST(DATE_FORMAT(t.transaction_date,'yyyyMMdd') 
             AS INT)                                    AS date_key,
        c.customer_id,
        p.product_id,
        s.store_id,
        t.order_id,
        t.order_line_num,
        t.quantity,
        t.unit_price,
        t.unit_cost,
        t.discount_amount,
        t.net_revenue,
        t.gross_profit,
        t.tax_amount
    FROM silver_transactions t
    LEFT JOIN silver_customers c ON t.customer_id = c.customer_id
    LEFT JOIN silver_products  p ON t.product_id  = p.product_id
    LEFT JOIN silver_stores    s ON t.store_id    = s.store_id
""")

df_gold_fact.write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3a://gold/delta/fact_sales")

print(f"Gold fact_sales: {df_gold_fact.count():,} rows")

✔ Gold fact_sales: 500,000 rows


In [ ]:
from pyspark.sql.functions import monotonically_increasing_id, lit
from datetime import date

# ─────────────────────────────────────────
# GOLD: DIM_CUSTOMER (SCD2 initial load)
# ─────────────────────────────────────────
df_gold_customers = spark.sql("""
    SELECT
        customer_id,
        customer_name,
        email,
        city,
        state,
        zip_code,
        customer_segment
    FROM silver_customers
""").withColumn("customer_key",   monotonically_increasing_id().cast("int") + lit(1)) \
   .withColumn("effective_date", lit(str(date(2024, 1, 1)))) \
   .withColumn("expiry_date",    lit(str(date(9999, 12, 31)))) \
   .withColumn("is_current",     lit(True))

df_gold_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3a://gold/delta/customers")

print(f"Gold customers : {df_gold_customers.count():,} rows")

# ─────────────────────────────────────────
# GOLD: DIM_PRODUCT
# ─────────────────────────────────────────
df_gold_products = spark.sql("""
    SELECT
        product_id,
        product_name,
        category,
        subcategory,
        brand,
        list_price,
        cost_price,
        is_active
    FROM silver_products
""").withColumn("product_key", monotonically_increasing_id().cast("int") + lit(1))

df_gold_products.write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3a://gold/delta/products")

print(f"Gold products  : {df_gold_products.count():,} rows")

# ─────────────────────────────────────────
# GOLD: DIM_STORE
# ─────────────────────────────────────────
df_gold_stores = spark.sql("""
    SELECT
        store_id,
        store_name,
        city,
        state,
        region,
        store_type,
        opening_date
    FROM silver_stores
""").withColumn("store_key", monotonically_increasing_id().cast("int") + lit(1))

df_gold_stores.write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3a://gold/delta/stores")

print(f"Gold stores    : {df_gold_stores.count():,} rows")

✔ Gold customers : 10,000 rows
✔ Gold products  : 1,000 rows
✔ Gold stores    : 200 rows


In [ ]:
# ─────────────────────────────────────────
# VERIFY GOLD DELTA TABLES
# ─────────────────────────────────────────
tables = ["fact_sales", "customers", "products", "stores"]

for table in tables:
    df = spark.read.format("delta").load(f"s3a://gold/delta/{table}")
    print(f"gold/delta/{table}: {df.count():,} rows | "
          f"{len(df.columns)} columns")

✔ gold/delta/fact_sales: 500,000 rows | 14 columns
✔ gold/delta/customers: 10,000 rows | 11 columns
✔ gold/delta/products: 1,000 rows | 9 columns
✔ gold/delta/stores: 200 rows | 8 columns
